<a href="https://colab.research.google.com/github/maliaheupel/ds2002-fa2/blob/main/03%20-%20pandas%20/%202026_09_25_%E2%80%94_Cleaning_Gauntlet_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [16]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [17]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [18]:
# TODO
removed = df.duplicated().sum()
clean = df.drop_duplicates().copy()

log('duplicates', 'dropped exact duplicate rows', removed)
clean.head()

[duplicates] dropped exact duplicate rows (15 row(s))


,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 2 — clean `price` -> float

In [19]:
# TODO
clean['price'] = clean['price'].astype(str).str.replace('$', '').str.strip().astype(float)
log('price', 'stripped dollar signs and converted price to float', len(clean))
print(clean['price'])

[price] stripped dollar signs and converted price to float (300 row(s))
0      12.0
1       7.5
2       7.5
3       7.5
4       7.5
       ... 
295     6.0
296     6.0
297     7.5
298     7.5
299    12.0
Name: price, Length: 300, dtype: float64


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [36]:
# TODO
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')
clean = clean.dropna(subset=['qty'])
clean = clean[clean['qty'] > 0]
log('qty', 'dropped missing and negative quantities', len(clean))
print(clean['qty'])

[qty] dropped missing and negative quantities (275 row(s))
0      3.0
1      1.0
2      1.0
4      1.0
5      3.0
      ... 
294    2.0
295    3.0
297    1.0
298    1.0
299    1.0
Name: qty, Length: 275, dtype: float64


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [21]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse
print(clean['item'].value_counts())
ITEM_MAP = {
    'cheese burger': 'Cheese Burger',
    'Cheese Burger': 'Cheese Burger',
    'cheeseburger': 'Cheese Burger',
    'foam finger': 'Foam Finger',
    'Foam Finger': 'Foam Finger',
    'rain poncho': 'Rain Poncho',
    'Rain Poncho': 'Rain Poncho',
}
clean['item'] = clean['item'].map(ITEM_MAP)
log('item', 'canonicalized item names', len(clean))
print(clean['item'])

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[item] canonicalized item names (275 row(s))
0        Rain Poncho
1        Foam Finger
2      Cheese Burger
4      Cheese Burger
5        Foam Finger
           ...      
294      Rain Poncho
295      Rain Poncho
297      Foam Finger
298      Rain Poncho
299    Cheese Burger
Name: item, Length: 275, dtype: object


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [22]:
# TODO
print(clean['category'].value_counts())
CATEGORY_MAP = {
    'Food': 'Food',
    'food': 'Food',
    'Merch': 'Merch',
    'Apparel': 'Merch',
    'Merchandise': 'Merch',
    'Rain Gear': 'Rain Gear',
    'RainGear': 'Rain Gear',
    'rain-gear': 'Rain Gear',
}
clean['category'] = clean['category'].map(CATEGORY_MAP)
log('category', 'canonicalized category names', len(clean))
print(clean['category'])

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[category] canonicalized category names (275 row(s))
0      Rain Gear
1          Merch
2          Merch
4          Merch
5           Food
         ...    
294        Merch
295         Food
297         Food
298        Merch
299         Food
Name: category, Length: 275, dtype: object


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [27]:
assert clean.duplicated().sum() == 0
assert clean['qty'].min() >= 1
assert clean['price'].dtype == float
# TODO: assert something about item
assert df['item'].isnull().sum() == 0
# TODO: assert something about category
assert set(clean['category'].unique()) == {'Food', 'Merch', 'Rain Gear'}
print('clean:', df.shape)

clean: (315, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [30]:
# TODO
clean['revenue'] = clean['qty'] * clean['price']
category_revenue = clean.groupby('category')['revenue'].sum().sort_values(ascending=False).round(2)

print("Revenue by Category:")
print(category_revenue)
print("\nTotal Revenue:", round(clean['revenue'].sum(), 2))


Revenue by Category:
category
Food         1656.0
Merch        1572.0
Rain Gear    1512.0
Name: revenue, dtype: float64

Total Revenue: 4740.0


**What I would tell the vendor:** _..._
I would advise vendors to stock more Food, since it generated the highest revenue comapred to the other categories.

### TODO 8 — read back your log

In [31]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,dropped exact duplicate rows,15
1,price,stripped dollar signs and converted price to f...,300
2,qty,dropped missing and negative quantities,275
3,item,canonicalized item names,275
4,category,canonicalized category names,275


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

In [41]:
# revenue before cleaning
raw_qty = pd.to_numeric(df['qty'], errors='coerce')
raw_price = pd.to_numeric(df['price'].astype(str).str.replace('$', ''), errors='coerce')
rev_raw = (raw_qty * raw_price).sum()

# final cleaned revenue
rev_clean = (clean['qty'] * clean['price']).sum()

print("Revenue before cleaning (raw data):", rev_raw)
print("Revenue after cleaning (final clean):", rev_clean)

Revenue before cleaning (raw data): 4852.5
Revenue after cleaning (final clean): 4740.0


_your answer here_

a) Dropping the negative or invalid quantities and missing values in the qty column changed the revenue total the most. The revenue before the drop was 4,852.50 and afterwards it was 4,740.00.

b) For step 5, I decided to merge Apparel into Merch. An alternative would have been to keep them as separate categories, since it would not have an affect on the revenue total. I chose to merge them because clothing items are part of merchandise, and combining them gives a wider view of how products are performing.